# Results for model: nvidia/llama-3.1-nemotron-nano-8b-v1

In [ ]:
import polars as p
import xgboost as xgb

# Load the data with Polars
df = p.read_table('./jane_street_data/train.parquet')

# Split into features and target
features = p.select_columns(['feature_00', 'responder_6'])
target = p.select_column('responder_6')
dates = p.select_column('date_id')

# Function to calculate global TOP_QUANTILE (top 15%) of feature_00 relative to responder_6
def get_quantile(dates, feature_00, responder_6, top_percent=15):
    return responder_6.quantile(top_percent)

# Rolling batches
window_size = 1  # Assuming 1 day for simplicity, adjust as needed
batch_size = 1
batches = []
for i in range(0, len(dates), batch_size):
    batch_dates = dates.iloc[i:i+batch_size]
    batch_feature_00 = features.iloc[i:i+batch_size, 0]
    batch_responder_6 = features.iloc[i:i+batch_size, 1]
    batch_feature_00_relative = batch_feature_00 / batch_responder_6  # Relative feature
    quantile_value = get_quantile(batch_responder_6, batch_feature_00_relative, None, top_percent)
    quantile_value = batch_feature_00.quantile().quantile(top_percent)  # Original feature
    batches.append((batch_dates, batch_feature_00, batch_responder_6, quantile_value))

# Now create the TOP_QUANTILE_00 feature column
top_quantile_00 = p.Series()
top_quantile_00.name = 'TOP_QUANTILE_00'
for batch in batches:
    batch_dates, batch_feature_00, batch_responder_6, quantile_value = batch
    # Here we add quantile_from_batch, but to keep it as a single scalar for comparison across days
    # We can't directly add to the column, so instead append the quantile to the Series
    top_quantile_00 = top_quantile_00.append(quantile_value)
top_quantile_00 = top_quantile_00.reset_index()

# Merge back into features
features = p.merge(features, top_quantile_00, on=df.index)

# Prepare XGB model
model = xgb.XGBRegressor(random_state=42)
model.fit(xgboost.DatasetsRegressor.from_pandas(features, target))

# The model object is now ready for predictions